In [1]:
import warnings
from rdkit import RDLogger

# 屏蔽 RDKit 警告
RDLogger.DisableLog('rdApp.*')

# 或屏蔽所有 Python 警告
warnings.filterwarnings("ignore")
# 屏蔽 LightGBM 警告
warnings.filterwarnings("ignore", category=UserWarning, module="lightgbm")

In [2]:
import torch
from sklearn.model_selection import StratifiedKFold
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import AllChem
from sklearn.metrics import precision_recall_curve, auc
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import joblib
import optuna
from rdkit.Chem import Descriptors, AllChem
from tqdm import tqdm  # 导入tqdm
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GroupKFold





In [3]:
# 函数：将SMILES转换为分子描述符和指纹
def smiles_to_features(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    # 提取描述符
    descriptors = [
        Descriptors.MolWt(mol),  # 分子量
        Descriptors.MolLogP(mol),  # LogP
        Descriptors.NumHDonors(mol),  # 氢键供体数量
        Descriptors.NumHAcceptors(mol)  # 氢键受体数量
    ]
    # 生成Morgan指纹
    fingerprint = AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=2048)
    fingerprint_array = np.zeros((2048,))
    Chem.DataStructs.ConvertToNumpyArray(fingerprint, fingerprint_array)
    # 合并描述符和指纹
    features = np.concatenate([descriptors, fingerprint_array])
    return features


In [10]:

from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import GroupKFold
from tqdm import tqdm
import optuna
import numpy as np

def train_evaluate_regression_model_with_optuna(model_name, model_class, param_func, X, y, groups):
    def objective(trial):
        params = param_func(trial)
        model = model_class(**params)

        gkf = GroupKFold(n_splits=10)
        maes = []

        for train_idx, val_idx in tqdm(gkf.split(X, y, groups=groups), total=10, desc=f"Training {model_name}"):
            X_train, X_val = X[train_idx], X[val_idx]
            y_train, y_val = y[train_idx], y[val_idx]

            model.fit(X_train, y_train)
            y_pred = model.predict(X_val)

            # ✅ 计算 MAE
            mae = mean_absolute_error(y_val, y_pred)
            maes.append(mae)

        return np.mean(maes)

    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=30)

    print(f'Best parameters for {model_name}: {study.best_params}')
    print(f'Best mean MAE: {study.best_value:.4f}')

    

In [4]:
# 数据预处理
df = pd.read_excel('../invertebrates_EC50_unique.xlsx')
labels = df['mgperL'].values
smiles_list = df['SMILES_Canonical_RDKit'].tolist()
endpoints_a = df['endpoint']
Duration_Values_a = df['Duration_Value'].values
effects_a = df['effect']


In [5]:

features = []
new_labels = []
new_smiles_list = []
endpoints = []
Duration_Values = []
effects =[]


for smiles, label,a,b,c in zip(smiles_list, labels,Duration_Values_a,effects_a,endpoints_a):
    feature = smiles_to_features(smiles)
    if feature is not None:
        features.append(feature)
        new_labels.append(label)
        new_smiles_list.append(smiles)
        Duration_Values.append(a)
        effects.append(b)
        endpoints.append(c)

X = np.array(features)
y = np.array(new_labels)
groups = new_smiles_list  # 可直接用于 GroupKFold




In [6]:
def encode_column(zz):
    zz_series = pd.Series(zz)  # 转换为 Series
    unique_values = zz_series.unique()
    if len(unique_values) > 1:
        encoder = OneHotEncoder(sparse_output=False)
        return encoder.fit_transform(zz_series.values.reshape(-1, 1))
    else:
        return None  # 只有一种类别时忽略

Duration_Values =pd.Series(Duration_Values)


# 编码 effect、endpoint 和 species_group 列
effect_encoded = encode_column(effects)
endpoint_encoded = encode_column(endpoints)
#species_encoded = encode_column(df, 'species_group')

# # 将需要的列拼接成输入 X
X = np.hstack((X, Duration_Values.values.reshape(-1, 1)))

# # 拼接编码后的列（如果存在）
for encoded_feature in [effect_encoded, endpoint_encoded]:
     if encoded_feature is not None:
         X = np.hstack((X, encoded_feature))



y=np.log1p(y)

In [11]:
def xgb_param_func(trial):
    return {
        'n_estimators': trial.suggest_int('n_estimators', 100, 600),
        'max_depth': trial.suggest_int('max_depth', 5, 20),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 1.0),   # L1 正则
        'reg_lambda': trial.suggest_float('reg_lambda', 0.0, 1.0)  # L2 正则
    }
from xgboost import XGBRegressor

train_evaluate_regression_model_with_optuna(
    "XGBoost",
    XGBRegressor,
    xgb_param_func,
    X, y, groups
)

[I 2025-05-15 17:42:27,725] A new study created in memory with name: no-name-8919a704-7695-4b72-a6a1-e3e97bde83ba
Training XGBoost: 100%|██████████| 10/10 [04:26<00:00, 26.70s/it]
[I 2025-05-15 17:46:54,711] Trial 0 finished with value: 0.9617998825906353 and parameters: {'n_estimators': 587, 'max_depth': 20, 'learning_rate': 0.011847034565104293, 'subsample': 0.6545279116403966, 'colsample_bytree': 0.7858482582958551, 'reg_alpha': 0.4650399271439267, 'reg_lambda': 0.908785571359224}. Best is trial 0 with value: 0.9617998825906353.
Training XGBoost: 100%|██████████| 10/10 [04:17<00:00, 25.74s/it]
[I 2025-05-15 17:51:12,102] Trial 1 finished with value: 0.9782800812727134 and parameters: {'n_estimators': 431, 'max_depth': 19, 'learning_rate': 0.01007540092638327, 'subsample': 0.9392021792518999, 'colsample_bytree': 0.7333677260794539, 'reg_alpha': 0.0031913661654645598, 'reg_lambda': 0.06859817275311664}. Best is trial 0 with value: 0.9617998825906353.
Training XGBoost: 100%|██████████|

Best parameters for XGBoost: {'n_estimators': 599, 'max_depth': 15, 'learning_rate': 0.058543399924681695, 'subsample': 0.7586019774543268, 'colsample_bytree': 0.9270215070117838, 'reg_alpha': 0.22301923725247752, 'reg_lambda': 0.4138408997745912}
Best mean MAE: 0.9483


In [12]:
from lightgbm import LGBMRegressor

def lgbm_param_func(trial):
    return {
        'n_estimators': trial.suggest_int('n_estimators', 100, 600),
        'max_depth': trial.suggest_int('max_depth', 5, 20),
        'num_leaves': trial.suggest_int('num_leaves', 20, 300),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.6, 1.0),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.6, 1.0),
        'bagging_freq': trial.suggest_int('bagging_freq', 1, 7),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 1.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.0, 1.0),
        'verbose': -1
    }

print("Training LightGBM (Poisson)...")
train_evaluate_regression_model_with_optuna(
    "LightGBM",
    lambda **params: LGBMRegressor(objective="poisson", **params),  # ✅ 加入 Poisson 目标
    lgbm_param_func,
    X, y, groups
)

[I 2025-05-15 18:50:26,653] A new study created in memory with name: no-name-7545c53b-6812-4a55-9f34-b2e8d3231811


Training LightGBM (Poisson)...


Training LightGBM: 100%|██████████| 10/10 [00:20<00:00,  2.05s/it]
[I 2025-05-15 18:50:47,221] Trial 0 finished with value: 1.0074074690214716 and parameters: {'n_estimators': 324, 'max_depth': 13, 'num_leaves': 222, 'learning_rate': 0.039283272012913026, 'feature_fraction': 0.7706304379577787, 'bagging_fraction': 0.9386470082073242, 'bagging_freq': 7, 'reg_alpha': 0.3499923130101006, 'reg_lambda': 0.6478267186174118}. Best is trial 0 with value: 1.0074074690214716.
Training LightGBM: 100%|██████████| 10/10 [00:16<00:00,  1.64s/it]
[I 2025-05-15 18:51:03,610] Trial 1 finished with value: 0.9978967720002849 and parameters: {'n_estimators': 263, 'max_depth': 20, 'num_leaves': 115, 'learning_rate': 0.29391116969051434, 'feature_fraction': 0.8207857652798478, 'bagging_fraction': 0.6020462110505491, 'bagging_freq': 6, 'reg_alpha': 0.7048290700335436, 'reg_lambda': 0.4670268346289508}. Best is trial 1 with value: 0.9978967720002849.
Training LightGBM: 100%|██████████| 10/10 [00:13<00:00,  1.

Best parameters for LightGBM: {'n_estimators': 551, 'max_depth': 18, 'num_leaves': 140, 'learning_rate': 0.0981145051227407, 'feature_fraction': 0.943191993958653, 'bagging_fraction': 0.9624771089477884, 'bagging_freq': 3, 'reg_alpha': 0.3274727963693285, 'reg_lambda': 0.31734500687536715}
Best mean MAE: 0.9432


In [9]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error
import optuna
import numpy as np


class DNNWithSoftplus(nn.Module):
    def __init__(self, input_dim, hidden_sizes, activation):
        super().__init__()
        act_fn = {
            'relu': nn.ReLU(),
            'logistic': nn.Sigmoid(),
            'tanh': nn.Tanh()
        }[activation]
        layers = []
        prev_dim = input_dim
        for h in hidden_sizes:
            layers += [nn.Linear(prev_dim, h), act_fn]
            prev_dim = h
        layers += [nn.Linear(prev_dim, 1)]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return F.softplus(self.net(x)).squeeze(-1)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
def train_dnn_with_optuna_pytorch(X, y, groups, device=device):
    def dnn_param_func(trial):
        return {
            'hidden_layer_sizes': trial.suggest_categorical(
                'hidden_layer_sizes', [(50,), (100,), (150,), (100, 50), (150, 100, 50)]
            ),
            'activation': trial.suggest_categorical('activation', ['relu', 'logistic', 'tanh']),
            'alpha': trial.suggest_float('alpha', 1e-5, 1e-2, log=True),
            'learning_rate': trial.suggest_float('learning_rate_init', 1e-4, 1e-2, log=True),
            'optimizer': trial.suggest_categorical('solver', ['adam', 'sgd'])
        }

    def objective(trial):
        params = dnn_param_func(trial)
        model = DNNWithSoftplus(
            input_dim=X.shape[1],
            hidden_sizes=params['hidden_layer_sizes'],
            activation=params['activation']
        ).to(device)

        optimizer = {
            'adam': torch.optim.Adam,
            'sgd': torch.optim.SGD
        }[params['optimizer']](model.parameters(), lr=params['learning_rate'], weight_decay=params['alpha'])

        loss_fn = nn.MSELoss()
        gkf = GroupKFold(n_splits=10)
        fold_maes = []

        for train_idx, val_idx in gkf.split(X, y, groups=groups):
            X_train, y_train = X[train_idx], y[train_idx]
            X_val, y_val = X[val_idx], y[val_idx]

            scaler = StandardScaler()
            X_train = scaler.fit_transform(X_train)
            X_val = scaler.transform(X_val)

            train_ds = TensorDataset(torch.tensor(X_train).float(), torch.tensor(y_train).float())
            train_loader = DataLoader(train_ds, batch_size=256, shuffle=True)

            model.train()
            for epoch in range(100):
                for xb, yb in train_loader:
                    xb, yb = xb.to(device), yb.to(device)
                    optimizer.zero_grad()
                    pred = model(xb)
                    loss = loss_fn(pred, yb)
                    loss.backward()
                    optimizer.step()

            model.eval()
            with torch.no_grad():
                val_preds = model(torch.tensor(X_val).float().to(device)).cpu().numpy()
                mae = mean_absolute_error(y_val, val_preds)
                fold_maes.append(mae)

        return np.mean(fold_maes)

    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=30)
    print("\n✅ Best Parameters Found:")
    print(study.best_params)
    print(f"Mean MAE = {study.best_value:.4f}")
    return study.best_params


best_dnn_params = train_dnn_with_optuna_pytorch(X, y, groups)

[I 2025-05-15 16:17:24,159] A new study created in memory with name: no-name-32f76744-7904-4494-b8ea-81aa7da190bb
[I 2025-05-15 16:19:20,116] Trial 0 finished with value: 1.1152298583636826 and parameters: {'hidden_layer_sizes': (150,), 'activation': 'logistic', 'alpha': 0.007583376930760382, 'learning_rate_init': 0.008670988225845841, 'solver': 'adam'}. Best is trial 0 with value: 1.1152298583636826.
[I 2025-05-15 16:21:11,130] Trial 1 finished with value: 0.4937479579514979 and parameters: {'hidden_layer_sizes': (50,), 'activation': 'relu', 'alpha': 1.5539826171203388e-05, 'learning_rate_init': 0.0009444402015257628, 'solver': 'adam'}. Best is trial 1 with value: 0.4937479579514979.
[I 2025-05-15 16:23:09,754] Trial 2 finished with value: 0.8309428881084434 and parameters: {'hidden_layer_sizes': (100, 50), 'activation': 'logistic', 'alpha': 0.0004841412288558556, 'learning_rate_init': 0.0016041838183389041, 'solver': 'adam'}. Best is trial 1 with value: 0.4937479579514979.
[I 2025-05


✅ Best Parameters Found:
{'hidden_layer_sizes': (150, 100, 50), 'activation': 'relu', 'alpha': 1.0259711717638815e-05, 'learning_rate_init': 0.00029459482163368076, 'solver': 'adam'}
Mean MAE = 0.4122
